# Attention Seq2seqによる計算機作成

## 目的
Attention Seq2seqの構造を理解する

## Attention機構
Seq2Seqはエンコーダが長期のパターンを学習する際に直近の情報に強く影響されるため，過去の特徴をうまく捉えることが難しいとされています．そこで，各時刻のエンコーダの出力を保持し，デコーダ側へ情報を伝搬するAttention機構を導入することで，この問題を解決します．Attention機構は保持したエンコーダの出力をデコーダの出力に対して重みづけすることで，どの時刻のエンコーダに着目してデコーダが文字等の情報を生成したかを可視化することもできます．


## モジュールのインポート

はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import random
import numpy as np
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの作成

計算式（足し算）のデータセット（`CalcDataset`）は`seq2seq.ipynb`と同様のものを使用します．

In [ ]:
word2id = {str(i): i for i in range(10)}
word2id.update({"<pad>": 10, "+": 11, "<eos>": 12})
id2word = {v: k for k, v in word2id.items()}


class CalcDataset(torch.utils.data.Dataset):

    # 計算式の文字列をIDに変換するための関数
    def transform(self, string, seq_len=7):
        tmp = []
        for i, c in enumerate(string):
            try:
                tmp.append(word2id[c])
            except KeyError:
                tmp += [word2id["<pad>"]] * (seq_len - i)
                break
        return tmp

    def __init__(self, data_num, train=True):
        super().__init__()

        self.data_num = data_num
        self.train = train
        self.data = []
        self.label = []

        for _ in range(data_num):
            x = random.randint(0, 999)
            y = random.randint(0, 999)
            left = ("{:*<7s}".format(str(x) + "+" + str(y))).replace("*", "<pad>")
            self.data.append(self.transform(left))

            z = x + y
            right = ("{:*<6s}".format(str(z))).replace("*", "<pad>")
            right = self.transform(right, seq_len=5)
            right = [12] + right
            right[right.index(10)] = 12
            self.label.append(right)

        self.data = np.asarray(self.data)
        self.label = np.asarray(self.label)

    def __getitem__(self, item):
        d = self.data[item]
        l = self.label[item]
        return d, l

    def __len__(self):
        return self.data.shape[0]

## ネットワークモデル（計算機）の定義
基本的な構造は`seq2seq.ipynb`のエンコーダ・デコーダ構造と同様です．ただし，エンコーダは各時刻の出力値（`encoder_outputs`）を保持しておきます．デコーダでは，保持したエンコーダの出力値とデコーダの出力値で内積計算します．この内積計算によって，各時刻のエンコーダの出力値に重み付けすることができます．これにより，どの時刻のエンコーダの出力に着目したかをデコーダ側が自動で決定することができます．

入力は`<pad>`で固定長に揃えていますが，`<pad>`の部分は本来意味を持たない時刻です．そこで，エンコーダでは`pack_padded_sequence`を使って`<pad>`の時刻をLSTMの計算対象から除外し，デコーダのAttentionスコア計算でも`<pad>`に対応する位置のスコアを`-inf`にしてから`softmax`を取ることで，`<pad>`位置にAttentionの重みが乗らないようにしています．

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2id["<pad>"])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, input_indices):
        embedding = self.word_embeddings(input_indices)

        # <pad>を除いた実際の系列長を求め，pack_padded_sequenceでLSTMに<pad>の時刻を計算させないようにする
        input_lengths = (input_indices != word2id["<pad>"]).sum(dim=1).clamp(min=1).cpu()
        packed_embedding = nn.utils.rnn.pack_padded_sequence(
            embedding, input_lengths, batch_first=True, enforce_sorted=False
        )
        packed_outputs, state = self.lstm(packed_embedding)
        # <pad>の時刻を元の系列長（input_indices.size(1)）に合わせてゼロ埋めで復元する
        encoder_outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=input_indices.size(1)
        )

        return encoder_outputs, state, input_lengths


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2id["<pad>"])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim * 2, vocab_size)

        self.softmax = nn.Softmax(dim=1)

    def forward(self, decoder_input, encoder_outputs, state, input_lengths):
        embedding = self.word_embeddings(decoder_input)
        if embedding.dim() == 2:
            embedding = torch.unsqueeze(embedding, 1)
        decoder_outputs, state = self.lstm(embedding, state)

        # エンコーダの各時刻の出力とデコーダの各時刻の出力の内積からattention scoreを求める
        attention_score = torch.bmm(encoder_outputs, decoder_outputs.transpose(1, 2))

        # <pad>の位置のscoreを-infにして，softmax後の重みがほぼ0になるようにする
        input_len = encoder_outputs.size(1)
        pad_positions = torch.arange(input_len, device=encoder_outputs.device)[None, :] >= input_lengths[:, None].to(encoder_outputs.device)
        attention_score = attention_score.masked_fill(pad_positions.unsqueeze(2), float("-inf"))

        attention_weight = self.softmax(attention_score)

        # 各デコーダ時刻について，attention weightで重み付けしたエンコーダ出力（コンテキストベクトル）をまとめて計算する
        context_vector = torch.bmm(attention_weight.transpose(1, 2), encoder_outputs)

        output = torch.cat([decoder_outputs, context_vector], dim=2)
        output = self.output(output)
        return output, state, attention_weight

## ネットワークモデルの作成

上で定義したエンコーダとデコーダを作成します．
エンコーダとデコーダは別々のネットワークとして用意し，それぞれの最適化にはAdamを利用します．

In [ ]:
embedding_dim = 16
hidden_dim = 128
vocab_size = len(word2id)

encoder = Encoder(vocab_size, embedding_dim, hidden_dim).to(device)
decoder = Decoder(vocab_size, embedding_dim, hidden_dim).to(device)

# 最適化手法の設定
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)

## 学習
学習を行います．学習データの準備や誤差計算の基本的な流れは`seq2seq.ipynb`と同様です．ただし，本ノートブックのデコーダは，正解ラベル全体（`source`）を一度に受け取り，Attention機構による重み付けを経て，全時刻分の出力とattention weightをまとめて返す点が異なります．

In [ ]:
batch_size = 100
epoch_num = 100

# データローダの準備
train_data = CalcDataset(data_num=50000)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

# 誤差関数の設定
criterion = nn.CrossEntropyLoss(ignore_index=word2id["<pad>"]).to(device)

# ネットワークを学習モードへ変更
encoder.train()
decoder.train()


start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0

    for data, label in train_loader:
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        data = data.to(device)
        label = label.to(device)

        encoder_outputs, encoder_state, input_lengths = encoder(data)
        source = label[:, :-1]
        target = label[:, 1:]
        decoder_state = encoder_state

        loss = 0
        decoder_output, _, attention_weight = decoder(source, encoder_outputs, decoder_state, input_lengths)
        for j in range(decoder_output.size(1)):
            loss += criterion(decoder_output[:, j, :], target[:, j])

        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()

        sum_loss += loss.item()

    elapsed_time = time() - start
    if epoch % 10 == 0:
        print("epoch: {}, mean loss: {:.4f}, elapsed_time: {:.4f}".format(epoch, sum_loss / len(train_loader), elapsed_time))

    # 10 epochに1回ネットワークモデルのパラメータを保存
    if epoch % 10 == 0:
        model_name = "attention_calc_v{}.pt".format(epoch)
        torch.save({
            'encoder_model': encoder.state_dict(),
            'decoder_model': decoder.state_dict(),
        }, model_name)

## 評価

次に，学習したモデルを評価します．テストデータを50サンプル生成して，データローダに与えます．
評価には，実際にモデルを使って計算結果を得る場面を反映した「自己回帰的な複数ステップ先予測」を用います．

### GPUが使用できず，学習ができなかった場合

学習済みモデルを用意していますので，下記のコマンドを実行してファイルをダウンロードしてください．

In [ ]:
# データのダウンロード
if not os.path.isfile('attention_calc_v100.pt'):
    gdown.download(id='1qgfwDUKAuVEd9oFDSsZUevBEfG7xzw7N', output='attention_calc_v100.pt', quiet=False)

エンコーダ・デコーダはバッチサイズを固定せず，入力テンソルから毎回バッチサイズを求める実装になっているため，学習時（バッチサイズ100）とテスト時（バッチサイズ1）で同じクラスをそのまま使えます．

ここでは，学習済みパラメータを読み込むために，エンコーダとデコーダを新たに生成します．

In [ ]:
test_data = CalcDataset(data_num=50)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False)

encoder = Encoder(vocab_size, embedding_dim, hidden_dim).to(device)
decoder = Decoder(vocab_size, embedding_dim, hidden_dim).to(device)

epoch_num = 100
model_name = "attention_calc_v{}.pt".format(epoch_num)
checkpoint = torch.load(model_name, map_location=device)
encoder.load_state_dict(checkpoint["encoder_model"])
decoder.load_state_dict(checkpoint["decoder_model"])

encoder.eval()
decoder.eval()

### 自己回帰的な複数ステップ先予測
実際に計算式から答えを生成する場面では正解ラベルは手元にないため，デコーダは自分自身が予測した文字を次時刻の入力として使い回しながら，答えの文字列を1文字ずつ生成していく必要があります．ここでは，開始記号`<eos>`から始めて，デコーダ自身の予測結果を次時刻の入力として繰り返し与え，`<eos>`が出力されるまで生成を続けます．これが，実際にモデルを使って計算結果を得る場面を反映した，より現実的な評価です．

In [ ]:
accuracy_ar = 0

# 評価の実行
with torch.no_grad():
    for data, label in test_loader:
        data = data.to(device)

        encoder_outputs, state, input_lengths = encoder(data)

        # decoderの計算：直前の自分自身の予測結果を次時刻の入力として使い回す
        right = []
        token = "<eos>"
        for _ in range(7):
            index = word2id[token]
            input_tensor = torch.tensor([index], device=device)
            output, state, _ = decoder(input_tensor, encoder_outputs, state, input_lengths)
            prob = F.softmax(torch.squeeze(output), dim=0)
            index = torch.argmax(prob.cpu().detach()).item()
            token = id2word[index]
            if token == "<eos>":
                break
            right.append(token)
        right = "".join(right)

        if "+" in right or "<pad>" in right:
            continue

        x = list(data[0].cpu().detach().numpy())
        try:
            padded_idx_x = x.index(word2id["<pad>"])
        except ValueError:
            padded_idx_x = len(x)
        left = "".join(map(lambda c: str(id2word[c]), x[:padded_idx_x]))

        # 正解判定（rightが空文字などint変換できない場合はFalse扱いにする）
        try:
            flag = ["F", "T"][eval(left) == int(right)]
        except ValueError:
            flag = "F"
        print("{:>7s} = {:>4s} :{}".format(left, right, flag))
        if flag == "T":
            accuracy_ar += 1

print("Accuracy (autoregressive): {:.2f}".format(accuracy_ar / len(test_loader)))

## Attentionの可視化
Decoder内のAttention weightの可視化をします．Attention weightを見ることで，デコーダがどのエンコーダの入力に着目したかを確認することができます．Attention weightの可視化にはヒートマップがよく用いられるので，ヒートマップで可視化してみます．ただし，全ての評価サンプルを確認すると時間もかかるので，今回は5サンプルを実行するごとにランダム表示します．ヒートマップは縦軸がエンコーダの入力，横軸がデコーダの出力を表しています．1数字ずつ見たとき，左に並んでいるボックスの色が一番明るいところの文字が最も着目して生成された数値を表しています．プロット毎に数値をランダムにしているので，各自ヒートマップの結果を考察してみてください．

なお，モデル（`encoder`, `decoder`）は上の自己回帰的な複数ステップ先予測で読み込んだものをそのまま使用し，ここではランダムにサンプルを選ぶための`test_loader`のみ作り直します．

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=True)

accuracy = 0

# 評価の実行
with torch.no_grad():
    for ind, (data, label) in enumerate(test_loader):
        data = data.to(device)

        encoder_outputs, state, input_lengths = encoder(data)

        right = []
        Atten = []
        token = "<eos>"
        for _ in range(7):
            index = word2id[token]
            input_tensor = torch.tensor([index], device=device)
            output, state, attention_weight = decoder(input_tensor, encoder_outputs, state, input_lengths)
            prob = F.softmax(torch.squeeze(output), dim=0)
            index = torch.argmax(prob.cpu().detach()).item()
            token = id2word[index]
            if token == "<eos>":
                break
            right.append(token)
            Atten.append(attention_weight.cpu().detach().numpy())
        str_right = right
        right = "".join(right)

        # rightが空（1文字目でEOSを予測した場合）や記号・パディングを含む場合はスキップ
        if len(Atten) == 0 or "+" in right or "<pad>" in right:
          accuracy += 0
          continue

        x = list(data[0].cpu().detach().numpy())
        try:
            padded_idx_x = x.index(word2id["<pad>"])
        except ValueError:
            padded_idx_x = len(x)
        left = "".join(map(lambda c: str(id2word[c]), x[:padded_idx_x]))
        str_left = []
        for s in range(len(x)):
          if str(x[s]) == '11':
            str_left.append('+')
          elif str(x[s]) == '10':
            str_left.append('=')
          else:
            str_left.append(str(x[s]))

        # 正解判定（rightが空文字などint変換できない場合はFalse扱いにする）
        try:
            flag = ["F", "T"][eval(left) == int(right)]
        except ValueError:
            flag = "F"
        print("{:>7s} = {:>4s} :{}".format(left, right, flag))
        Atten = np.concatenate(Atten, axis=0)
        Atten = Atten[:, :, 0].transpose(1, 0)
        df = pd.DataFrame(Atten, index=str_left, columns=str_right)
        plt.figure(figsize=[12, 8])
        sns.heatmap(df)
        if ind == 4:
          break

## 課題

1. 足し算だけでなく，色々な四則演算を実装しましょう．
2. Attention weightのヒートマップを見比べて，デコーダが答えの各桁を生成する際に，実際にエンコーダのどの位置（数字や記号）に注目しているか考察してみましょう．
3. `hidden_dim`や`embedding_dim`を変更して学習し，計算精度やAttention weightの可視化結果にどのような変化が現れるか確認してみましょう．